# Silver Layer Transformation Pipeline - Generic & Metadata-Driven

Generic notebook to apply transformations, deduplicate, and merge Bronze data into Silver Delta tables.

In [ ]:
# Notebook Parameters
dbutils.widgets.text("config_path", "/Workspace/Users/jayarampogakula@gmail.com/lakeforge/configs/pipeline_config.json", "Config Path")
dbutils.widgets.text("source_table", "customers", "Source Bronze Table")
dbutils.widgets.text("target_table", "customers_clean", "Target Silver Table")
dbutils.widgets.dropdown("environment", "dev", ["dev", "prod"], "Environment")
dbutils.widgets.dropdown("use_scd_type2", "false", ["true", "false"], "Use SCD Type 2")

config_path = dbutils.widgets.get("config_path")
source_table = dbutils.widgets.get("source_table")
target_table = dbutils.widgets.get("target_table")
environment = dbutils.widgets.get("environment")
use_scd_type2 = dbutils.widgets.get("use_scd_type2").lower() == "true"

print(f"Executing Silver transformations from {source_table} to {target_table} (SCD2: {use_scd_type2})")

In [ ]:
# Imports & Path setup
import sys
sys.path.append("/Workspace/Users/jayarampogakula@gmail.com/lakeforge")

from lakeforge import (
    ConfigParser,
    SilverTransformer,
    SilverMergeEngine,
    create_scd_type2_handler,
    create_trust_engine
)

print("✅ Framework libraries loaded")

In [ ]:
# Load Configurations
config = ConfigParser.parse_monolithic_pipeline_config(config_path)
env_config = config["environments"][environment]

bronze_catalog = env_config["catalog"]
bronze_schema = env_config["bronze_schema"]
silver_catalog = env_config["catalog"]
silver_schema = env_config["silver_schema"]

full_source_table = f"{bronze_catalog}.{bronze_schema}.{source_table}"
full_target_table = f"{silver_catalog}.{silver_schema}.{target_table}"

print(f"Source: {full_source_table}")
print(f"Target: {full_target_table}")

In [ ]:
# Read Bronze Table
df_bronze = spark.table(full_source_table)
print(f"✅ Loaded Bronze table: {df_bronze.count()} records")

In [ ]:
# Run Transformations dynamically
trans_configs = config["silver_transformations"].get(target_table, {}).get("transformations", [])

df_silver = SilverTransformer.transform(df_bronze, trans_configs)
print(f"✅ Applied transformations. Schema is now: {df_silver.columns}")

In [ ]:
# Write / Merge to Silver layer
source_meta = config["sources"][source_table]
business_keys = source_meta["business_key"]
merge_strat = source_meta.get("merge_strategy", "upsert")

if use_scd_type2:
    print("ℹ️ Performing SCD Type 2 merge...")
    scd_handler = create_scd_type2_handler(spark)
    # Remove existing SCD columns to avoid double calculation
    clean_cols = [c for c in df_silver.columns if c not in ["effective_start_date", "effective_end_date", "is_current", "record_hash"]]
    df_silver_clean = df_silver.select(*clean_cols)
    
    merge_stats = scd_handler.merge_scd_type2(
        source_df=df_silver_clean,
        target_table=target_table,
        catalog=silver_catalog,
        schema=silver_schema,
        business_keys=business_keys
    )
else:
    print(f"ℹ️ Performing standard merge/write ({merge_strat})...")
    mode_val = "merge" if merge_strat == "upsert" else "append" if merge_strat == "append" else "overwrite"
    merge_stats = SilverMergeEngine.merge(
        spark=spark,
        df=df_silver,
        target_table=full_target_table,
        merge_keys=business_keys,
        mode=mode_val
    )

print(f"✅ Write complete: {merge_stats}")

In [ ]:
# Compute and Log Trust Score
trust_engine = create_trust_engine(spark)
trust_validations = [
    {
        "type": "row_count",
        "params": {
            "source_df": df_bronze,
            "target_df": df_silver,
            "tolerance_percent": 5.0
        }
    },
    {
        "type": "duplicate_explosion",
        "params": {
            "source_df": df_bronze,
            "target_df": df_silver,
            "key_columns": business_keys,
            "max_explosion_ratio": 1.5
        }
    }
]

trust_results = trust_engine.run_trust_validations(trust_validations)
trust_score = trust_engine.calculate_trust_score(
    table_name=full_target_table,
    dq_results=trust_results,
    pipeline_stage="silver"
)

print(f"🎯 Pipeline Trust Score: {trust_score['overall_score']:.1f}%")
print(f"🎯 Trust Level: {trust_score['trust_level']}")